# Team 03G - IFQ580 Assignment 1

Team members
- Minchul (Michael) Kim
- Dmitry Novik
- Jane Norris

## Task 1: Data preparation for modelling (3.5 marks)
1. The dataset may include irrelevant and redundant variables to the underlying ML task. What variables did you include in the modelling, and what were their roles and measurement level set? Justify your choice.
2. Did you have to fix any data quality problems, including data imputation? Detail them.
3. Report the proportion of values of the target variable for the dataset before and after the pre-processing.

Load Libraries and Dataset

In [1]:
import pandas as pd
import numpy as np

# Load dataset (low_memory=False avoids DtypeWarning for mixed type columns)
df = pd.read_csv('kick_1.csv', low_memory=False)

print(f"Original dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Original dataset shape: 41476 rows, 31 columns


,PurchaseID,PurchaseTimestamp,PurchaseDate,Auction,VehYear,Make,Color,Transmission,WheelTypeID,WheelType,...,MMRCurrentRetailCleanPrice,MMRCurrentRetailRatio,PRIMEUNIT,AUCGUART,VNST,VehBCost,IsOnlineSale,WarrantyCost,ForSale,IsBadBuy
0,0,1253232000,18/09/2009 10:00,OTHER,2008.0,DODGE,RED,AUTO,2,Covers,...,12505,0.941783287,?,?,NC,7800,0,920.0,Yes,0
1,1,1253232000,18/09/2009 10:00,OTHER,2008.0,DODGE,RED,AUTO,2,Covers,...,10571,0.922618485,?,?,NC,7800,0,834.0,Yes,0
2,2,1253232000,18/09/2009 10:00,OTHER,2008.0,CHRYSLER,SILVER,AUTO,2,Covers,...,9932,0.935159082,?,?,NC,7800,0,834.0,Yes,0
3,3,1253232000,18/09/2009 10:00,OTHER,2008.0,CHEVROLET,RED,AUTO,2,Covers,...,8739,0.931456688,?,?,NC,6000,0,671.0,Yes,0
4,4,1253232000,18/09/2009 10:00,OTHER,2008.0,DODGE,SILVER,AUTO,2,Covers,...,9908,0.906943884,?,?,NC,7800,0,920.0,Yes,0


Check Target Variable Distribution Before Preprocessing

In [2]:
# Calculate target variable ('IsBadBuy') distribution before preprocessing
target_before = df['IsBadBuy'].value_counts(normalize=True) * 100
target_before_count = df['IsBadBuy'].value_counts()

summary_before = pd.DataFrame({
    'Count': target_before_count,
    'Percentage (%)': target_before.round(2)
})

print("=== [Before Preprocessing] Target Variable (IsBadBuy) Distribution ===")
print(summary_before)

=== [Before Preprocessing] Target Variable (IsBadBuy) Distribution ===
          Count  Percentage (%)
IsBadBuy                       
0         36105           87.05
1          5371           12.95


Data Cleaning (Replace Missing Markers and Convert Data Types)
Dropped Columns
- PurchaseID. It is a unique transaction key. Including unique identifiers causes models to memorize training samples (overfitting) rather than learning generalized patterns.
- PurchaseTimestamp. It provides the exact Unix timestamp, which is 100% redundant with PurchaseDate. Keeping both causes collinearity and adds unhelpful granularity.
- WheelTypeID. It is a duplicate numeric representation of WheelType. Retaining both creates redundant features without adding new information.
- ForSale. Over 99.9% of the entries are 'Yes', meaning it exhibits near-zero variance. Features with no variation across classes cannot aid the classifier in distinguishing kicks from normal cars.

In [3]:
# Drop redundant or non-informative columns from the working dataframe
cols_to_drop = ['PurchaseID', 'PurchaseTimestamp', 'WheelTypeID', 'ForSale']
df_cleaned = df.drop(columns=cols_to_drop)

# Define list of numeric price/valuation variables for classification
numeric_cols = [
    'VehOdo', 'VehBCost', 'WarrantyCost',
    'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice',
    'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice',
    'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice',
    'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice',
    'MMRCurrentRetailRatio'
]

# Generate comprehensive metadata table for ALL 31 original variables (Included & Excluded)
all_variable_roles = []

for col in df.columns:
    if col in cols_to_drop:
        status = 'Excluded (Dropped)'
        role = 'None'
        level = 'N/A'
        if col == 'PurchaseID':
            justification = 'Unique transaction identifier with high cardinality. Holds zero predictive capability.'
        elif col == 'PurchaseTimestamp':
            justification = '100% redundant with PurchaseDate. Exact timestamp introduces noise without predictive value.'
        elif col == 'WheelTypeID':
            justification = '100% duplicate numeric key of the categorical feature WheelType.'
        elif col == 'ForSale':
            justification = 'Near-zero variance feature (almost all values are Yes). Provides no discrimination power.'
    
    elif col == 'IsBadBuy':
        status = 'Included'
        role = 'Target (Output)'
        level = 'Binary / Nominal'
        justification = 'Binary classification target (1 = Kick / Bad Buy, 0 = Normal purchase).'
    
    elif col in numeric_cols:
        status = 'Included'
        role = 'Input Feature'
        level = 'Interval / Ratio (Continuous)'
        justification = 'Continuous metric representing vehicle mileage, purchase cost, warranty, or MMR benchmark valuation.'
    
    elif col == 'Size':
        status = 'Included'
        role = 'Input Feature'
        level = 'Ordinal'
        justification = 'Categorical feature with inherent logical ordering (e.g., COMPACT < MIDSIZE < LARGE).'
    
    elif col == 'VehYear':
        status = 'Included'
        role = 'Input Feature'
        level = 'Interval / Ratio'
        justification = 'Vehicle manufacturing year reflecting age-related depreciation and vehicle generation.'
    
    else:
        status = 'Included'
        role = 'Input Feature'
        level = 'Nominal (Categorical)'
        justification = 'Categorical attribute capturing auction house, make, color, region, or guarantee risk.'

    all_variable_roles.append({
        'Variable Name': col,
        'Status': status,
        'Role': role,
        'Measurement Level': level,
        'Justification': justification
    })

# Render complete justification table in Jupyter Notebook
justification_df = pd.DataFrame(all_variable_roles)

print(f"Total Original Variables: {len(df.columns)}")
print(f"Included Features for Modelling: {len(df_cleaned.columns)}")
print(f"Excluded Variables: {len(cols_to_drop)}")

# Display the entire table (set max_rows to display all 31 variables)
pd.set_option('display.max_rows', 35)
pd.set_option('display.max_colwidth', None)
justification_df

Total Original Variables: 31
Included Features for Modelling: 27
Excluded Variables: 4


,Variable Name,Status,Role,Measurement Level,Justification
0,PurchaseID,Excluded (Dropped),None,N/A,Unique transaction identifier with high cardinality. Holds zero predictive capability.
1,PurchaseTimestamp,Excluded (Dropped),None,N/A,100% redundant with PurchaseDate. Exact timestamp introduces noise without predictive value.
2,PurchaseDate,Included,Input Feature,Nominal (Categorical),"Categorical attribute capturing auction house, make, color, region, or guarantee risk."
3,Auction,Included,Input Feature,Nominal (Categorical),"Categorical attribute capturing auction house, make, color, region, or guarantee risk."
4,VehYear,Included,Input Feature,Interval / Ratio,Vehicle manufacturing year reflecting age-related depreciation and vehicle generation.
5,Make,Included,Input Feature,Nominal (Categorical),"Categorical attribute capturing auction house, make, color, region, or guarantee risk."
6,Color,Included,Input Feature,Nominal (Categorical),"Categorical attribute capturing auction house, make, color, region, or guarantee risk."
7,Transmission,Included,Input Feature,Nominal (Categorical),"Categorical attribute capturing auction house, make, color, region, or guarantee risk."
8,WheelTypeID,Excluded (Dropped),None,N/A,100% duplicate numeric key of the categorical feature WheelType.
9,WheelType,Included,Input Feature,Nominal (Categorical),"Categorical attribute capturing auction house, make, color, region, or guarantee risk."


Remove Irrelevant Features & Generate Variable Role Mapping Table

In [4]:
# Drop redundant or non-informative columns
cols_to_drop = ['PurchaseID', 'PurchaseTimestamp', 'WheelTypeID', 'ForSale']
df_cleaned = df.drop(columns=cols_to_drop)

# Generate metadata table detailing variable roles, measurement levels, and justifications
variable_roles = []
for col in df_cleaned.columns:
    if col == 'IsBadBuy':
        role = 'Target (Output)'
        level = 'Binary / Nominal'
        justification = 'Target label to predict (1: Kick/Bad Buy, 0: Normal)'
    elif col in numeric_cols:
        role = 'Input Feature'
        level = 'Interval / Ratio (Continuous)'
        justification = 'Continuous metrics covering vehicle condition, market valuation, and warranty costs'
    elif col == 'Size':
        role = 'Input Feature'
        level = 'Ordinal'
        justification = 'Vehicle size category with inherent ordinal progression (e.g., Compact < Midsize < Large)'
    else:
        role = 'Input Feature'
        level = 'Nominal (Categorical)'
        justification = 'Categorical attributes such as auction house, make, color, and region'
    
    variable_roles.append({
        'Variable Name': col,
        'Role': role,
        'Measurement Level': level,
        'Justification': justification
    })

role_df = pd.DataFrame(variable_roles)
print(f"Total remaining features: {len(df_cleaned.columns)}")
role_df.head(10)  # Render formatted summary table in Jupyter Notebook

Total remaining features: 27


,Variable Name,Role,Measurement Level,Justification
0,PurchaseDate,Input Feature,Nominal (Categorical),"Categorical attributes such as auction house, make, color, and region"
1,Auction,Input Feature,Nominal (Categorical),"Categorical attributes such as auction house, make, color, and region"
2,VehYear,Input Feature,Nominal (Categorical),"Categorical attributes such as auction house, make, color, and region"
3,Make,Input Feature,Nominal (Categorical),"Categorical attributes such as auction house, make, color, and region"
4,Color,Input Feature,Nominal (Categorical),"Categorical attributes such as auction house, make, color, and region"
5,Transmission,Input Feature,Nominal (Categorical),"Categorical attributes such as auction house, make, color, and region"
6,WheelType,Input Feature,Nominal (Categorical),"Categorical attributes such as auction house, make, color, and region"
7,VehOdo,Input Feature,Interval / Ratio (Continuous),"Continuous metrics covering vehicle condition, market valuation, and warranty costs"
8,Nationality,Input Feature,Nominal (Categorical),"Categorical attributes such as auction house, make, color, and region"
9,Size,Input Feature,Ordinal,"Vehicle size category with inherent ordinal progression (e.g., Compact < Midsize < Large)"


Missing Value Imputation

In [5]:
# Impute numerical missing values using feature Medians (robust against outliers)
num_vars = df_cleaned.select_dtypes(include=[np.number]).columns.drop('IsBadBuy')
for col in num_vars:
    median_val = df_cleaned[col].median()
    df_cleaned[col].fillna(median_val, inplace=True)

# Impute categorical missing values using feature Modes (most frequent value)
cat_vars = df_cleaned.select_dtypes(include=['object']).columns
for col in cat_vars:
    mode_val = df_cleaned[col].mode()[0]
    df_cleaned[col].fillna(mode_val, inplace=True)

print("Data imputation completed.")
print(f"Total remaining missing values: {df_cleaned.isnull().sum().sum()}")

Data imputation completed.
Total remaining missing values: 0


C:\Users\yuwol\AppData\Local\Temp\ipykernel_25480\876834638.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned[col].fillna(median_val, inplace=True)
C:\Users\yuwol\AppData\Local\Temp\ipykernel_25480\876834638.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

Verify Target Proportion After Preprocessing & Export Cleaned Dataset

In [6]:
# Calculate target variable ('IsBadBuy') distribution after preprocessing
target_after = df_cleaned['IsBadBuy'].value_counts(normalize=True) * 100
target_after_count = df_cleaned['IsBadBuy'].value_counts()

summary_after = pd.DataFrame({
    'Count (Before)': target_before_count,
    'Percentage Before (%)': target_before.round(2),
    'Count (After)': target_after_count,
    'Percentage After (%)': target_after.round(2)
})

print("=== Task 1 Comparison: Target Variable Distribution Before vs After ===")
print(summary_after)

# Export cleaned dataset for subsequent modelling tasks (Tasks 2-4)
df_cleaned.to_csv('kick_cleaned.csv', index=False)
print("\nDataset successfully saved to 'kick_cleaned.csv'.")

=== Task 1 Comparison: Target Variable Distribution Before vs After ===
          Count (Before)  Percentage Before (%)  Count (After)  \
IsBadBuy                                                         
0                  36105                  87.05          36105   
1                   5371                  12.95           5371   

          Percentage After (%)  
IsBadBuy                        
0                        87.05  
1                        12.95  



Dataset successfully saved to 'kick_cleaned.csv'.
